# RREF v6e smoke training

Small Colab/local notebook for the finite-field RREF stack. It checks the repo checkout, builds one fixed-shape shard, runs a short pivot-policy training job, performs neural rollout, and benchmarks leftmost plus neural policies.

Command surface used: `nf-agent --help`, `nf-agent data make-rref-shard`, `nf-agent train rref-pivot`, `nf-agent rollout rref-neural`, `nf-agent benchmark rref`.


## Colab boundary

- Runtime: Python 3.12. A TPU v6e runtime can run the JAX training cell when available, but these cells also run on CPU for smoke validation.
- Checkout: run this notebook from the repository root after cloning the repo.
- Teacher shard generation is explicit offline CPU work. Neural rollout invokes the trained policy with legal action masking only.
- Outputs go to `/tmp`; no repo-tracked artifacts are created by the cells.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
SHARD_PATH = Path("/tmp/rref_8x8_train_smoke.npz")
CKPT_DIR = Path("/tmp/rref_pivot_ckpt")
HIDDEN_SIZE = "32"


def run_cmd(args: list[str], *, cwd: Path = REPO_ROOT) -> subprocess.CompletedProcess[str]:
    print("$ " + " ".join(args))
    result = subprocess.run(
        args,
        cwd=cwd,
        check=True,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr[-2000:])
    return result


def reset_tmp_path(path: Path) -> None:
    resolved = path.resolve()
    if not resolved.is_relative_to(Path("/tmp").resolve()):
        raise RuntimeError(f"refusing to remove non-/tmp path: {resolved}")
    if resolved.is_dir():
        shutil.rmtree(resolved)
    elif resolved.exists():
        resolved.unlink()


print({"repo_root": str(REPO_ROOT), "python": sys.version, "platform": platform.platform()})
assert (REPO_ROOT / "pyproject.toml").is_file(), "run from the repo root"
assert (REPO_ROOT / "src" / "nf_agent").is_dir(), "src/nf_agent missing"
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("nf-agent requires Python 3.12")

if shutil.which("nf-agent") is None or os.environ.get("NF_AGENT_INSTALL") == "1":
    run_cmd([sys.executable, "-m", "pip", "install", "-r", "requirements-dev.txt", "-e", "."])

run_cmd([sys.executable, "-m", "pip", "check"])
help_result = run_cmd(["nf-agent", "--help"])
assert "Verifier-guided normal-form agent tools" in help_result.stdout


## Generate a tiny RREF shard

The shard uses `configs/rref_8x8_mod101.yaml`, which selects dense 8x8 matrices over `F_101` and the explicit leftmost teacher as the dataset source.


In [ ]:
import numpy as np

reset_tmp_path(SHARD_PATH)
run_cmd(
    [
        "nf-agent",
        "data",
        "make-rref-shard",
        "--config",
        "configs/rref_8x8_mod101.yaml",
        "--count",
        "8",
        "--seed-start",
        "0",
        "--out",
        str(SHARD_PATH),
    ]
)

with np.load(SHARD_PATH, allow_pickle=False) as shard:
    metadata = json.loads(str(shard["metadata_json"]))
    print(json.dumps(metadata, indent=2, sort_keys=True))
    print({"inputs_shape": shard["inputs"].shape, "op_kind_shape": shard["op_kind"].shape})


## Train a small pivot policy

Two steps and a single 32-wide hidden layer keep the smoke run short. Use the same `--hidden-size 32` in rollout and benchmark so checkpoint shapes match.


In [ ]:
reset_tmp_path(CKPT_DIR)
train_result = run_cmd(
    [
        "nf-agent",
        "train",
        "rref-pivot",
        "--data",
        str(SHARD_PATH),
        "--steps",
        "2",
        "--batch-size",
        "4",
        "--learning-rate",
        "0.001",
        "--seed",
        "0",
        "--hidden-size",
        HIDDEN_SIZE,
        "--out",
        str(CKPT_DIR),
    ]
)
train_payload = json.loads(train_result.stdout)
print(
    json.dumps(
        {
            "status": train_payload["status"],
            "latest_step": train_payload["latest_step"],
            "final_step": train_payload["final_step"],
            "parameters_changed": train_payload["parameters_changed"],
            "checkpoint_dir": train_payload["checkpoint_dir"],
        },
        indent=2,
        sort_keys=True,
    )
)


## Neural rollout

Replay one shard sample with the trained policy and print only the compact status fields plus verification flags.


In [ ]:
rollout_result = run_cmd(
    [
        "nf-agent",
        "rollout",
        "rref-neural",
        "--data",
        str(SHARD_PATH),
        "--checkpoint",
        str(CKPT_DIR),
        "--sample-index",
        "0",
        "--max-steps",
        "8",
        "--hidden-size",
        HIDDEN_SIZE,
    ]
)
rollout_payload = json.loads(rollout_result.stdout)
rollout_summary_keys = [
    "status",
    "success",
    "step_count",
    "invalid_action_count",
    "masked_action_count",
    "final_is_rref",
    "checkpoint_step",
    "modulus",
]
print(json.dumps({key: rollout_payload[key] for key in rollout_summary_keys}, indent=2))


## Benchmark leftmost and neural policies

Shard-source benchmarking always runs the exact leftmost baseline. Providing the checkpoint also adds the neural policy on the same shard samples.


In [ ]:
benchmark_result = run_cmd(
    [
        "nf-agent",
        "benchmark",
        "rref",
        "--source",
        "shard",
        "--data",
        str(SHARD_PATH),
        "--count",
        "4",
        "--checkpoint",
        str(CKPT_DIR),
        "--max-steps",
        "8",
        "--hidden-size",
        HIDDEN_SIZE,
    ]
)
benchmark_payload = json.loads(benchmark_result.stdout)
policy_summary = {
    name: {
        "sample_count": policy["aggregate"]["sample_count"],
        "success_rate": policy["aggregate"]["success_rate"],
        "status_counts": policy["aggregate"]["status_counts"],
    }
    for name, policy in benchmark_payload["policies"].items()
}
print(json.dumps(policy_summary, indent=2, sort_keys=True))


## Extension

For a longer run, increase shard `--count`, training `--steps`, and benchmark `--count` together. Keep checkpoint and rollout `--hidden-size` values identical to the training cell.
